# Uno et al. 2005 Optik: Digitized Aberrations

This notebook computes the digitized aberration quantities defined by Uno et al., *Optik* 116 (2005), formulas (38)-(47), using simulated probe profiles from the Colab GPU smoke-test workflow.

Reference used for the formulas: local PDF `/Users/yuemingguo/Downloads/Uno et al 2005 Optik.pdf`, page 445.

Important notation: Uno's profile width is written with the Greek letter sigma in formula (45). In this notebook it is named `Xigma` so it is not confused with the Gaussian probe smearing parameter `sigma=2` used during probe image generation.


## 1. Check GPU


In [73]:
!nvidia-smi


Thu May 28 04:37:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             29W /   70W |     219MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone or Pull Latest GitHub Code


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)


Repository already exists. Pulling latest main...
repo root: /content/Aberration-Simulation
commit: 5a99823


: 

## 3. Install and Verify Dependencies


In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")


Dependencies are ready.


: 

## 4. Verify CuPy Backend


In [ ]:
!python scripts/check_backend.py

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU smoke path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())


HAS_CUPY: True
backend module: cupy
backend version: 14.0.1
cuda runtime: 12090
gpu device: Tesla T4
CuPy GPU smoke path is active.
device count: 1


: 

## 5. Run Probe Simulation

This uses the same smoke-test simulation path as `notebooks/colab_gpu_smoke_test.ipynb`.


In [ ]:
!python scripts/run_smoke_test.py


backend: cupy
parameter combinations: 326
parameter families: {'baseline': 4, 'c3': 20, 'a1': 128, 'a2': 50, 'b2': 124}
probe image shape: (256, 256, 326)
intensity range: 7.140125527318166e-10 0.0012661891216764376
saved: /content/Aberration-Simulation/outputs/smoke_probe_images.npz
saved: /content/Aberration-Simulation/outputs/smoke_parameters.csv


: 

## 6. Uno Formula Implementation

From the local PDF, formulas (38)-(44):

- `Cdf_value = (1/N) sum_k (Xigma_u,k - Xigma_o,k)`
- `A1_value = (2/N) sum_k (Xigma_u,k - Xigma_o,k) exp(2 i theta_k)`
- `B2_value = (2/N) sum_k (Mu_u,k + Mu_o,k) exp(i theta_k)`
- `A2_value = (2/N) sum_k (Mu_u,k + Mu_o,k) exp(3 i theta_k)`
- `Cs_value = (1/N) sum_k (Rho_u,k - Rho_o,k)`
- `S3_value = (2/N) sum_k (Rho_u,k - Rho_o,k) exp(2 i theta_k)`
- `A3_value = (2/N) sum_k (Xigma_u,k - Xigma_o,k) exp(4 i theta_k)`

Formulas (45)-(47) define line-profile characteristics from profile samples `p_j`, where `j=0` is the probe center, `W=sum_j p_j`, and `T=sum_j p_j^2`:

- `Xigma = sqrt((1/W) sum_j j^2 p_j)`
- `Mu = (1/W) sum_j j p_j`
- `Rho = (Xigma^2/T) sum_{j != 0} ((p_j - p_0) p_j / abs(j))`


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aberration_simulation.line_profiles import extract_line_profiles_from_stack

OUTPUT_DIR = ROOT / "outputs"
NPZ_PATH = OUTPUT_DIR / "smoke_probe_images.npz"
CSV_PATH = OUTPUT_DIR / "uno_2005_digitized_aberrations.csv"

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

print("under-focus C1_offset:", UNDER_FOCUS_C1_OFFSET, "nm")
print("over-focus C1_offset:", OVER_FOCUS_C1_OFFSET, "nm")
print("line-profile angles: 0 to 170 degrees in 10-degree counter-clockwise steps")


under-focus C1_offset: -909 nm
over-focus C1_offset: 909 nm
line-profile angles: 0 to 170 degrees in 10-degree steps


: 

In [ ]:
def load_smoke_outputs(path):
    data = np.load(path, allow_pickle=True)
    names = [str(name) for name in data["parameter_names"]]
    rows = data["parameters"]
    parameters = [
        {name: float(value) for name, value in zip(names, row)}
        for row in rows
    ]
    return data["probe_images"], parameters


COMBINATION_FIELDS = (
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)


def combination_key(params):
    return tuple(params.get(field, 0.0) for field in COMBINATION_FIELDS)


def select_under_over_pairs(parameters):
    pairs = {}
    representatives = {}
    for index, params in enumerate(parameters):
        key = combination_key(params)
        representatives.setdefault(key, params)
        pair = pairs.setdefault(key, {})
        if np.isclose(params["C1_offset"], UNDER_FOCUS_C1_OFFSET):
            pair["under"] = index
        if np.isclose(params["C1_offset"], OVER_FOCUS_C1_OFFSET):
            pair["over"] = index

    selected = []
    for key, pair in pairs.items():
        if "under" in pair and "over" in pair:
            selected.append((representatives[key], pair["under"], pair["over"]))
    return selected


probe_images, parameters = load_smoke_outputs(NPZ_PATH)
pairs = select_under_over_pairs(parameters)

print("probe image stack:", probe_images.shape)
print("under/over pairs:", len(pairs))


probe image stack: (256, 256, 326)
under/over pairs: 162


: 

In [ ]:
def compute_line_characteristics(profiles_np, radius):
    """Compute Xigma, Mu, and Rho for each angular line profile.

    profiles_np has shape `(num_angles, 2 * radius + 1)`.
    The profile-coordinate index j is measured in pixels, with j=0 at center.
    """
    j = np.arange(-radius, radius + 1, dtype=float)
    center_index = int(np.argmin(np.abs(j)))
    p0 = profiles_np[:, center_index]

    W = np.sum(profiles_np, axis=1)
    T = np.sum(profiles_np ** 2, axis=1)
    W = np.where(W == 0, np.nan, W)
    T = np.where(T == 0, np.nan, T)

    Xigma = np.sqrt(np.sum((j[None, :] ** 2) * profiles_np, axis=1) / W)
    Mu = np.sum(j[None, :] * profiles_np, axis=1) / W

    nonzero = j != 0
    curvature_sum = np.sum(
        ((profiles_np[:, nonzero] - p0[:, None]) * profiles_np[:, nonzero])
        / np.abs(j[nonzero])[None, :],
        axis=1,
    )
    Rho = (Xigma ** 2 / T) * curvature_sum

    return {"Xigma": Xigma, "Mu": Mu, "Rho": Rho}


def compute_uno_values(under_chars, over_chars, angles_deg):
    theta = np.deg2rad(angles_deg)
    N = len(theta)

    Xigma_diff = under_chars["Xigma"] - over_chars["Xigma"]
    Mu_sum = under_chars["Mu"] + over_chars["Mu"]
    Rho_diff = under_chars["Rho"] - over_chars["Rho"]

    Cdf_value = np.sum(Xigma_diff) / N
    A1_value = 2 * np.sum(Xigma_diff * np.exp(2j * theta)) / N
    B2_value = 2 * np.sum(Mu_sum * np.exp(1j * theta)) / N
    A2_value = 2 * np.sum(Mu_sum * np.exp(3j * theta)) / N
    Cs_value = np.sum(Rho_diff) / N
    S3_value = 2 * np.sum(Rho_diff * np.exp(2j * theta)) / N
    A3_value = 2 * np.sum(Xigma_diff * np.exp(4j * theta)) / N

    return {
        "Cdf_value": Cdf_value,
        "A1_value": A1_value,
        "B2_value": B2_value,
        "A2_value": A2_value,
        "Cs_value": Cs_value,
        "S3_value": S3_value,
        "A3_value": A3_value,
    }


UNO_HARMONIC_ORDERS = {
    "A1_value": 2,
    "B2_value": 1,
    "A2_value": 3,
    "S3_value": 2,
    "A3_value": 4,
}

# Fitted by `notebooks/uno_et_al_2005_optik_auto_convention_search.ipynb`.
# All terms use the positive raw harmonic phase divided by harmonic order;
# A1 and A3 need the expected 180/order offset, while B2, A2, and S3 do not.
PRIMARY_PHASE_CONVENTIONS = {
    "A1_value": {"sign": 1.0, "offset_deg": 90.0},
    "B2_value": {"sign": 1.0, "offset_deg": 0.0},
    "A2_value": {"sign": 1.0, "offset_deg": 0.0},
    "S3_value": {"sign": 1.0, "offset_deg": 0.0},
    "A3_value": {"sign": 1.0, "offset_deg": 45.0},
}


def wrap_period_deg(angle_deg, period_deg):
    return float(np.mod(angle_deg, period_deg))


def harmonic_orientation_deg(phase_deg, order):
    return wrap_period_deg(phase_deg / order, 360.0 / order)


def interpreted_harmonic_phase_deg(raw_complex_phase_deg, order, convention):
    period_deg = 360.0 / order
    return wrap_period_deg(
        convention["sign"] * raw_complex_phase_deg / order + convention["offset_deg"],
        period_deg,
    )


def add_complex_columns(row, name, value):
    raw_phase_deg = float(np.angle(value, deg=True))
    row[name] = str(value)
    row[name + "_real"] = float(np.real(value))
    row[name + "_imag"] = float(np.imag(value))
    row[name + "_abs"] = float(np.abs(value))
    row[name + "_complex_phase_deg"] = raw_phase_deg
    row[name + "_raw_phase_deg"] = raw_phase_deg

    order = UNO_HARMONIC_ORDERS.get(name)
    if order is not None:
        period_deg = 360.0 / order
        convention = PRIMARY_PHASE_CONVENTIONS[name]
        orientation_deg = harmonic_orientation_deg(raw_phase_deg, order)
        orientation_negated_deg = harmonic_orientation_deg(-raw_phase_deg, order)
        orientation_sign_flipped_deg = harmonic_orientation_deg(raw_phase_deg + 180.0, order)
        orientation_negated_sign_flipped_deg = harmonic_orientation_deg(-(raw_phase_deg + 180.0), order)
        interpreted_phase_deg = interpreted_harmonic_phase_deg(raw_phase_deg, order, convention)

        row[name + "_orientation_deg"] = orientation_deg
        row[name + "_orientation_negated_deg"] = orientation_negated_deg
        row[name + "_orientation_sign_flipped_deg"] = orientation_sign_flipped_deg
        row[name + "_orientation_negated_sign_flipped_deg"] = orientation_negated_sign_flipped_deg
        row[name + "_orientation_period_deg"] = period_deg
        row[name + "_phase_convention"] = "sign={sign:g}, offset={offset:g} deg".format(
            sign=convention["sign"],
            offset=convention["offset_deg"],
        )
        row[name + "_phase_sign"] = float(convention["sign"])
        row[name + "_phase_offset_deg"] = float(convention["offset_deg"])
        row[name + "_phase_period_deg"] = period_deg
        row[name + "_phase_deg"] = interpreted_phase_deg
        row[name + "_interpreted_phase_deg"] = interpreted_phase_deg
        row[name + "_interpreted_phase_period_deg"] = period_deg
    else:
        row[name + "_phase_deg"] = raw_phase_deg


## 7. Compute Uno Digitized Aberrations


In [ ]:
rows = []
characteristics_by_case = []

for case_index, (params, under_index, over_index) in enumerate(pairs):
    stack = probe_images[:, :, [under_index, over_index]]
    profiles, coords = extract_line_profiles_from_stack(
        stack,
        num_lines=NUM_PROFILE_LINES,
        radius=PROFILE_RADIUS_PIXELS,
    )
    profiles_np = asnumpy(profiles)
    angles_deg = np.asarray(coords["angles_deg"], dtype=float)

    under_chars = compute_line_characteristics(profiles_np[:, :, 0], PROFILE_RADIUS_PIXELS)
    over_chars = compute_line_characteristics(profiles_np[:, :, 1], PROFILE_RADIUS_PIXELS)
    uno_values = compute_uno_values(under_chars, over_chars, angles_deg)

    row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
    row["case_index"] = case_index
    row["under_index"] = under_index
    row["over_index"] = over_index
    row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
    row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
    for name, value in uno_values.items():
        add_complex_columns(row, name, value)
    rows.append(row)

    characteristics_by_case.append({
        "params": params,
        "angles_deg": angles_deg,
        "under_chars": under_chars,
        "over_chars": over_chars,
        "uno_values": uno_values,
    })

print("computed cases:", len(rows))
print("first case Uno values:")
for key, value in characteristics_by_case[0]["uno_values"].items():
    print("  ", key, value)


computed cases: 162
first case Uno values:
   Cdf_value 5.921189464667501e-16
   A1_value (-4.156806925292844e-16-9.287931712876556e-16j)
   B2_value (1.1782921068107013e-15+2.849833450131683e-15j)
   A2_value (1.5793021197640236e-15-1.1206344062203412e-15j)
   Cs_value 5.477100254817439e-15
   S3_value (-2.135057775810891e-15-4.0746066429686027e-16j)
   A3_value (1.4180839279937027e-15+1.3501107129093427e-16j)


: 

## 8. Save Results


In [ ]:
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = list(rows[0].keys())
with CSV_PATH.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("saved:", CSV_PATH)
print("columns:", len(fieldnames))


saved: /content/Aberration-Simulation/outputs/uno_2005_digitized_aberrations.csv
columns: 85


: 

## 9. Interpret and Visualize Harmonic Phases

The scalar Uno quantities `Cdf_value` and `Cs_value` do not have angular phase conventions. The harmonic quantities `A1_value`, `B2_value`, `A2_value`, `S3_value`, and `A3_value` do: their raw complex phases are stored as `*_complex_phase_deg` / `*_raw_phase_deg`, while the primary reported `*_phase_deg` columns apply the fitted physical orientation convention for displayed probe images. Line-profile angles increase counter-clockwise in display coordinates. The fitted convention from `uno_et_al_2005_optik_auto_convention_search.ipynb` is `raw_complex_phase / harmonic_order + offset`, with offsets `A1=90 deg`, `B2=0 deg`, `A2=0 deg`, `S3=0 deg`, and `A3=45 deg`.


In [ ]:
def circular_difference_deg(a, b, period):
    return np.abs((a - b + period / 2) % period - period / 2)


ORIENTATION_CANDIDATES = [
    "orientation_deg",
    "orientation_negated_deg",
    "orientation_sign_flipped_deg",
    "orientation_negated_sign_flipped_deg",
]

SCALAR_UNO_COLUMNS = [
    "Cdf_value",
    "Cs_value",
]

PHASE_DIAGNOSTIC_SPECS = [
    {
        "label": "A1",
        "value_name": "A1_value",
        "amp_field": "A1_amp",
        "phase_field": "A1_phase",
    },
    {
        "label": "B2/C21",
        "value_name": "B2_value",
        "amp_field": "B2_amp",
        "phase_field": "B2_phase",
    },
    {
        "label": "A2",
        "value_name": "A2_value",
        "amp_field": "A2_amp",
        "phase_field": "A2_phase",
    },
    {
        "label": "A3",
        "value_name": "A3_value",
        "amp_field": "A3_amp",
        "phase_field": "A3_phase",
    },
    {
        "label": "S3/C32",
        "value_name": "S3_value",
        "amp_field": "S3_amp",
        "phase_field": "S3_phase",
    },
]

PRIMARY_PHASE_COLUMNS = [
    value_name + "_phase_deg"
    for value_name in UNO_HARMONIC_ORDERS
]

RAW_PHASE_COLUMNS = [
    value_name + "_complex_phase_deg"
    for value_name in UNO_HARMONIC_ORDERS
]

PRIMARY_PHASE_PLOT_PATH = OUTPUT_DIR / "uno_primary_phase_vs_input_all_swept_harmonic_coefficients.png"
CONVENTION_ERROR_PLOT_PATH = OUTPUT_DIR / "uno_phase_convention_candidate_errors.png"


def build_phase_diagnostics(rows, spec):
    value_name = spec["value_name"]
    amp_field = spec["amp_field"]
    phase_field = spec["phase_field"]
    order = UNO_HARMONIC_ORDERS[value_name]
    period = 360.0 / order

    diagnostics = []
    for row in rows:
        if np.isclose(row.get(amp_field, 0.0), 0.0):
            continue
        target = row[phase_field] % period
        candidates = {
            "orientation_deg": row[value_name + "_orientation_deg"],
            "orientation_negated_deg": row[value_name + "_orientation_negated_deg"],
            "orientation_sign_flipped_deg": row[value_name + "_orientation_sign_flipped_deg"],
            "orientation_negated_sign_flipped_deg": row[value_name + "_orientation_negated_sign_flipped_deg"],
            "primary_phase_deg": row[value_name + "_phase_deg"],
        }
        errors = {
            name + "_error": circular_difference_deg(value, target, period)
            for name, value in candidates.items()
        }
        diagnostics.append({
            "label": spec["label"],
            "value_name": value_name,
            "case_index": row["case_index"],
            "amp": row[amp_field],
            "input_phase_deg": row[phase_field],
            "input_phase_mod_period_deg": target,
            "period_deg": period,
            "phase_convention": row[value_name + "_phase_convention"],
            "complex_phase_deg_raw": row[value_name + "_complex_phase_deg"],
            **candidates,
            **errors,
        })
    return diagnostics


def summarize_phase_diagnostics(diagnostics):
    summary = []
    if not diagnostics:
        return summary
    for name in ORIENTATION_CANDIDATES + ["primary_phase_deg"]:
        errors = np.asarray([item[name + "_error"] for item in diagnostics], dtype=float)
        summary.append({
            "candidate": name,
            "mean_abs_error_deg": float(np.nanmean(errors)),
            "median_abs_error_deg": float(np.nanmedian(errors)),
            "max_abs_error_deg": float(np.nanmax(errors)),
        })
    return sorted(summary, key=lambda item: item["median_abs_error_deg"])


phase_diagnostics_by_coeff = {}
phase_summaries_by_coeff = {}
for spec in PHASE_DIAGNOSTIC_SPECS:
    diagnostics = build_phase_diagnostics(rows, spec)
    phase_diagnostics_by_coeff[spec["label"]] = diagnostics
    phase_summaries_by_coeff[spec["label"]] = summarize_phase_diagnostics(diagnostics)

print("Scalar Uno columns with no angular phase convention:")
for column in SCALAR_UNO_COLUMNS:
    print("  ", column)

print("Primary fitted phase conventions for harmonic Uno columns:")
for value_name, convention in PRIMARY_PHASE_CONVENTIONS.items():
    print("  {}_phase_deg: sign={:g}, offset={:g} deg".format(
        value_name,
        convention["sign"],
        convention["offset_deg"],
    ))
print("Raw complex phase columns retained in CSV:")
for column in RAW_PHASE_COLUMNS:
    print("  ", column)

print()
print("Phase convention diagnostics by swept input coefficient:")
for spec in PHASE_DIAGNOSTIC_SPECS:
    label = spec["label"]
    diagnostics = phase_diagnostics_by_coeff[label]
    summary = phase_summaries_by_coeff[label]
    order = UNO_HARMONIC_ORDERS[spec["value_name"]]
    period = 360.0 / order
    print("{}: {} nonzero cases, phase period {} deg".format(label, len(diagnostics), period))
    if not summary:
        print("  no nonzero sweep cases available in this smoke-test grid")
        continue
    print("  primary convention:", PRIMARY_PHASE_CONVENTIONS[spec["value_name"]])
    for item in summary:
        print("  ", item)

active_specs = [
    spec for spec in PHASE_DIAGNOSTIC_SPECS
    if phase_diagnostics_by_coeff[spec["label"]]
]

if active_specs:
    print("Rendering: Primary reported phase vs input phase for all swept harmonic coefficients")
    fig, axes = plt.subplots(1, len(active_specs), figsize=(4.8 * len(active_specs), 4.2), squeeze=False)
    for axis, spec in zip(axes.ravel(), active_specs):
        diagnostics = phase_diagnostics_by_coeff[spec["label"]]
        period = diagnostics[0]["period_deg"]
        target = np.asarray([item["input_phase_mod_period_deg"] for item in diagnostics])
        primary = np.asarray([item["primary_phase_deg"] for item in diagnostics])
        amp = np.asarray([item["amp"] for item in diagnostics])
        sc = axis.scatter(target, primary, c=amp, cmap="viridis", s=28, alpha=0.9)
        axis.plot([0, period], [0, period], color="black", linewidth=1, linestyle="--")
        pad = 0.04 * period
        axis.set_title(spec["label"])
        axis.set_xlabel("input phase mod period (deg)")
        axis.set_ylabel(spec["value_name"] + "_phase_deg")
        axis.set_xlim(-pad, period + pad)
        axis.set_ylim(-pad, period + pad)
        axis.grid(alpha=0.3)
        fig.colorbar(sc, ax=axis, label=spec["amp_field"])
    fig.suptitle("Primary reported phase vs input phase for all swept harmonic coefficients")
    fig.tight_layout()
    fig.savefig(PRIMARY_PHASE_PLOT_PATH, dpi=180, bbox_inches="tight")
    print("saved primary phase plot:", PRIMARY_PHASE_PLOT_PATH)
    plt.show()

if active_specs:
    print("Rendering: Uno phase convention candidate errors")
    candidate_names = ORIENTATION_CANDIDATES + ["primary_phase_deg"]
    fig, axes = plt.subplots(len(active_specs), 1, figsize=(10, 3.2 * len(active_specs)), squeeze=False)
    for axis, spec in zip(axes.ravel(), active_specs):
        summary_lookup = {
            item["candidate"]: item
            for item in phase_summaries_by_coeff[spec["label"]]
        }
        labels = [name.replace("orientation_", "").replace("_deg", "") for name in candidate_names]
        medians = [summary_lookup[name]["median_abs_error_deg"] for name in candidate_names]
        means = [summary_lookup[name]["mean_abs_error_deg"] for name in candidate_names]
        x = np.arange(len(labels))
        axis.bar(x - 0.18, medians, width=0.36, label="median error")
        axis.bar(x + 0.18, means, width=0.36, label="mean error")
        axis.set_xticks(x)
        axis.set_xticklabels(labels, rotation=25, ha="right")
        axis.set_ylabel("absolute error (deg)")
        axis.set_title(spec["label"] + " convention candidates")
        axis.grid(axis="y", alpha=0.3)
        axis.legend()
    fig.tight_layout()
    fig.savefig(CONVENTION_ERROR_PLOT_PATH, dpi=180, bbox_inches="tight")
    print("saved convention error plot:", CONVENTION_ERROR_PLOT_PATH)
    plt.show()


## 9. Quick Visual Check


In [ ]:
def find_first_case(**criteria):
    for index, item in enumerate(characteristics_by_case):
        params = item["params"]
        if all(np.isclose(params.get(key, 0.0), value) for key, value in criteria.items()):
            return index, item
    return 0, characteristics_by_case[0]


case_index, item = find_first_case(A1_amp=60, A1_phase=157.5)
angles_deg = item["angles_deg"]
row = rows[case_index]

fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
for axis, metric in zip(axes, ["Xigma", "Mu", "Rho"]):
    axis.plot(angles_deg, item["under_chars"][metric], marker="o", label="under")
    axis.plot(angles_deg, item["over_chars"][metric], marker="s", label="over")
    axis.set_ylabel(metric)
    axis.grid(True, alpha=0.3)
    axis.legend()
axes[-1].set_xlabel("line angle (deg)")
fig.suptitle("Uno line characteristics, case {}".format(case_index))
fig.tight_layout()
plt.show()

print("case parameters:", item["params"])
print("Uno values:")
for key, value in item["uno_values"].items():
    primary_phase = row.get(key + "_phase_deg")
    raw_phase = row.get(key + "_complex_phase_deg")
    period = row.get(key + "_phase_period_deg")
    print("  {} = {} | abs={} primary_phase_deg={} raw_complex_phase_deg={} period={}".format(
        key,
        value,
        np.abs(value),
        primary_phase,
        raw_phase,
        period,
    ))


: 